# 0.1. Ustawienie ścieżek

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve() / "src"))

from config.config import setup

cfg = setup()

HAR_ROOT = cfg["HAR_ROOT"]
MODELS_DIR = cfg["MODELS_DIR"]
RESULTS_DIR = cfg["RESULTS_DIR"]
FIGURES_DIR = cfg["FIGURES_DIR"]

# 0.2. Wgranie danych z datasetu

In [2]:
from data.loader import load_har_data

X_train, y_train, groups_train, X_test, y_test, groups_test, feature_names = (
    load_har_data(HAR_ROOT)
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

X_train: (7352, 561) | X_test: (2947, 561)


In [3]:
import numpy as np
from sklearn.model_selection import GroupKFold
from src.tuning.grid_search_cv import run_grid
from src.config.constants import N_JOBS

y_train_arr = np.asarray(y_train)
groups_train_arr = np.asarray(groups_train)

CV = GroupKFold(n_splits=5)
SCORING = "matthews_corrcoef"
ALL_SUMMARIES = []

# 1. Najlepsze hiperparametry z fazy 3

In [4]:
BEST_PARAMS_PHASE3 = {
    "dummy": {"clf__strategy": "uniform"},
    "logreg": {"clf__C": 1.0, "clf__solver": "newton-cg"},
    "linear_svc": {"clf__C": 0.1, "clf__loss": "squared_hinge"},
    "rbf_svc": {"clf__C": 10.0, "clf__gamma": 0.001},
    "random_forest": {
        "clf__max_depth": None,
        "clf__max_features": "sqrt",
        "clf__min_samples_split": 10,
        "clf__n_estimators": 300,
    },
    "gaussian_nb": {"clf__var_smoothing": 1e-05},
}

# 2. Stworzenie pipelinów dla najlepszych modeli

In [5]:
from models.pipelines import pipe_linear_svc, pipe_logreg, pipe_rbf_svc

PIPE_FACTORIES = {
    "linear_svc": pipe_linear_svc,
    "logreg": pipe_logreg,
    "rbf_svc": pipe_rbf_svc,
}


def make_tuned_factory(name, base_factory):
    best = BEST_PARAMS_PHASE3.get(name, {})

    def factory():
        pipe = base_factory()
        if best:
            pipe.set_params(**best)
        return pipe

    return factory


for name, base in PIPE_FACTORIES.items():
    p = make_tuned_factory(name, base)()
    print(f"[{name:<15}] {p.named_steps['clf']}")

[linear_svc     ] LinearSVC(C=0.1, max_iter=10000, random_state=42)
[logreg         ] LogisticRegression(max_iter=5000, random_state=42, solver='newton-cg')
[rbf_svc        ] SVC(C=10.0, gamma=0.001, random_state=42)


# 3. Permutation importance

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.inspection import permutation_importance
from src.config.constants import RANDOM_STATE
import time

vt = VarianceThreshold(threshold=0.0).fit(X_train)
X_train_vt = vt.transform(X_train)
mask_vt = vt.get_support()
kept_feature_names = np.array(feature_names)[mask_vt]
print(f"Po VarianceThreshold: {X_train_vt.shape[1]} cech (z {X_train.shape[1]}).")

rf_for_imp = RandomForestClassifier(
    n_estimators=500,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

t0 = time.perf_counter()
rf_for_imp.fit(X_train_vt, y_train_arr)
print(f"RF.fit: {time.perf_counter() - t0:.1f}s")

t0 = time.perf_counter()
perm_res = permutation_importance(
    rf_for_imp,
    X_train_vt,
    y_train_arr,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring=SCORING,
    n_jobs=N_JOBS,
)
print(f"permutation_importance: {time.perf_counter() - t0:.1f}s")

perm_mean = perm_res.importances_mean
perm_order = np.argsort(perm_mean)[::-1]
print("\nTop 10 cech wg permutation importance:")
for rank, idx in enumerate(perm_order[:10], 1):
    print(f"  {rank:2d}. {kept_feature_names[idx]:50s}  imp={perm_mean[idx]:.4f}")

Po VarianceThreshold: 561 cech (z 561).
RF.fit: 3.8s
permutation_importance: 220.0s

Top 10 cech wg permutation importance:
   1. angle(Z,gravityMean)                                imp=0.0000
   2. tBodyGyroJerk-entropy()-Y                           imp=0.0000
   3. tBodyGyroJerk-arCoeff()-Y,1                         imp=0.0000
   4. tBodyGyroJerk-arCoeff()-X,4                         imp=0.0000
   5. tBodyGyroJerk-arCoeff()-X,3                         imp=0.0000
   6. tBodyGyroJerk-arCoeff()-X,2                         imp=0.0000
   7. tBodyGyroJerk-arCoeff()-X,1                         imp=0.0000
   8. tBodyGyroJerk-entropy()-Z                           imp=0.0000
   9. tBodyGyroJerk-entropy()-X                           imp=0.0000
  10. tBodyGyroJerk-arCoeff()-Y,3                         imp=0.0000


In [7]:
import joblib

PERM_DIR = MODELS_DIR / "04_phase_permutation_importance"
PERM_DIR.mkdir(parents=True, exist_ok=True)

PERM_PATH = PERM_DIR / "permutation_importance.joblib"
joblib.dump(
    {
        "mask_vt": mask_vt,
        "kept_feature_names": kept_feature_names,
        "importances_mean": perm_mean,
        "order_desc": perm_order,
        "rf_for_imp": str(rf_for_imp),
    },
    PERM_PATH,
)

['/Users/aleks/PycharmProjects/JupyterProjectHar/models/04_phase_permutation_importance/permutation_importance.joblib']

# 4. PermutationTopK - klasa będąca selektorem do Pipeline

In [8]:
from sklearn.base import BaseEstimator, TransformerMixin


class PermutationTopK(BaseEstimator, TransformerMixin):
    def __init__(self, order=None, k=50):
        self.order = order
        self.k = k

    def fit(self, X, y=None):
        if self.order is None:
            raise ValueError("PermutationTopK: brak `order` (rankingu cech).")
        if self.k > len(self.order):
            raise ValueError(
                f"k={self.k} > liczba cech w rankingu ({len(self.order)})."
            )
        self.selected_idx_ = np.asarray(self.order[: self.k])
        return self

    def transform(self, X):
        return X[:, self.selected_idx_]

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.array([f"x{i}" for i in self.selected_idx_])
        return np.asarray(input_features)[self.selected_idx_]

# 5. Stworzenie wrapperów do selektorów, dodanie podmiany w pipelines placeholderów selector na właściwe selektory

In [9]:
from functools import partial

from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

K_GRID = [20, 50, 100, 200, 400]
N_COMP_GRID = [10, 25, 50, 100, 200]


def make_kbest_mi():
    return SelectKBest(
        score_func=partial(mutual_info_classif, random_state=RANDOM_STATE),
        k=50,
    )


def make_pca_095():
    return PCA(n_components=0.95, random_state=RANDOM_STATE)


def make_pca_tuned():
    return PCA(n_components=50, random_state=RANDOM_STATE)


def make_perm_topk():
    return PermutationTopK(order=perm_order, k=50)


def make_pipe_with_selector(tuned_factory, selector):
    pipe: Pipeline = tuned_factory()
    if "selector" in dict(pipe.steps):
        pipe.set_params(selector=selector)
        return pipe
    steps = list(pipe.steps)
    idx = [i for i, (n, _) in enumerate(steps) if n == "variance"][0]
    steps.insert(idx + 1, ("selector", selector))
    return Pipeline(steps)


def run_variant(
    model_name, base_factory, variant_name, selector, param_grid, subdir=None
):
    full_name = f"{model_name}__{variant_name}"
    tuned_factory = make_tuned_factory(model_name, base_factory)

    def factory():
        return make_pipe_with_selector(tuned_factory, selector)

    grid, summary = run_grid(
        full_name,
        factory,
        param_grid,
        X=X_train,
        y=y_train_arr,
        groups=groups_train_arr,
        models_dir=MODELS_DIR,
        results_dir=RESULTS_DIR,
        cv=CV,
        scoring=SCORING,
        n_jobs=N_JOBS,
        subdir=subdir,
    )
    summary["model"] = model_name
    summary["variant"] = variant_name
    ALL_SUMMARIES.append(summary)
    return grid, summary

# 6. Pętla

In [10]:
SUBDIR = "04_phase"
ALL_SUMMARIES.clear()

for model_name, base_factory in PIPE_FACTORIES.items():
    print(f"\n{'=' * 70}\n>>> Model: {model_name}\n{'=' * 70}")
    sub = f"{SUBDIR}_{model_name}"

    run_variant(model_name, base_factory, "baseline", "passthrough", {}, sub)
    run_variant(
        model_name,
        base_factory,
        "kbest_mi",
        make_kbest_mi(),
        {"selector__k": K_GRID},
        sub,
    )
    run_variant(model_name, base_factory, "pca_095", make_pca_095(), {}, sub)
    run_variant(
        model_name,
        base_factory,
        "pca_tuned",
        make_pca_tuned(),
        {"selector__n_components": N_COMP_GRID},
        sub,
    )
    run_variant(
        model_name,
        base_factory,
        "perm_topk",
        make_perm_topk(),
        {"selector__k": K_GRID},
        sub,
    )


>>> Model: linear_svc
Fitting 5 folds for each of 1 candidates, totalling 5 fits

[linear_svc__baseline]	Best MCC = 0.9258		|		1 kombinacji × 5 foldów = 5 fitów		|		czas = 15.6s
[linear_svc__baseline]	Best_params: {}
Fitting 5 folds for each of 5 candidates, totalling 25 fits

[linear_svc__kbest_mi]	Best MCC = 0.9128		|		5 kombinacji × 5 foldów = 25 fitów		|		czas = 194.0s
[linear_svc__kbest_mi]	Best_params: {'selector__k': 400}
Fitting 5 folds for each of 1 candidates, totalling 5 fits

[linear_svc__pca_095]	Best MCC = 0.8958		|		1 kombinacji × 5 foldów = 5 fitów		|		czas = 1.9s
[linear_svc__pca_095]	Best_params: {}
Fitting 5 folds for each of 5 candidates, totalling 25 fits

[linear_svc__pca_tuned]	Best MCC = 0.9111		|		5 kombinacji × 5 foldów = 25 fitów		|		czas = 8.9s
[linear_svc__pca_tuned]	Best_params: {'selector__n_components': 200}
Fitting 5 folds for each of 5 candidates, totalling 25 fits

[linear_svc__perm_topk]	Best MCC = 0.9287		|		5 kombinacji × 5 foldów = 25 fitów		|		c

# 7. Tabela

In [11]:
import pandas as pd

df_summ = pd.DataFrame(ALL_SUMMARIES)[
    ["model", "variant", "best_score", "best_params", "n_combos", "n_fits", "time_s"]
]
df_summ = df_summ.sort_values(
    ["model", "best_score"], ascending=[True, False]
).reset_index(drop=True)

pivot_mcc = df_summ.pivot(index="model", columns="variant", values="best_score")[
    ["baseline", "kbest_mi", "pca_095", "pca_tuned", "perm_topk"]
].round(4)
print("MCC (mean CV) per model × wariant:")
print(pivot_mcc)

SUMMARY_DIR = RESULTS_DIR / "04_phase_summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
df_summ.to_csv(SUMMARY_DIR / "summary.csv", index=False)

PIVOT_DIR = RESULTS_DIR / "04_phase_pivot_mcc"
PIVOT_DIR.mkdir(parents=True, exist_ok=True)
pivot_mcc.to_csv(PIVOT_DIR / "pivot_mcc.csv")

MCC (mean CV) per model × wariant:
variant     baseline  kbest_mi  pca_095  pca_tuned  perm_topk
model                                                        
linear_svc    0.9258    0.9128   0.8958     0.9111     0.9287
logreg        0.9227    0.9072   0.9027     0.9171     0.9238
rbf_svc       0.9177    0.9020   0.8988     0.9154     0.9229
